<a href="https://colab.research.google.com/github/kumarprakhar14/Agentic_AI_Course/blob/main/Module%203%20-%20OpenAI%20Agents%20SDK%20-%20Multi%20Agents/4.%20Build%20Multi%20AI%20Agents%20Workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TASK 1. PROJECT OVERVIEW AND KEY LEARNING OBJECTIVES

![image.png](attachment:6fe6d129-44d4-43a2-988e-d1fc077c6cec.png)

![image.png](attachment:40f2c461-e270-4c08-8ea0-186849eafc3c.png)

![image.png](attachment:9c637433-f564-4349-999f-45524f5e1c1e.png)

# TASK 2. SETUP API KEYS & DEFINE REQUIRED TOOLS

In [2]:
!pip install -q openai-agents==0.2.2 python-dotenv requests pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00


Configure OpenRouter for OpenAI Agent SDK

In [3]:
import os
import asyncio
from openai import AsyncOpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
from agents import set_default_openai_client, set_default_openai_api, set_tracing_disabled

# Load environment variables and configure client
load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# 1. Create a custom async client pointing to OpenRouter
or_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key
)

# 2. Inject it into the Agents SDK
set_default_openai_client(or_client)

# 3. CRITICAL: Force the SDK to use Chat Completions
# (OpenRouter does not support OpenAI's proprietary Responses API)
set_default_openai_api("chat_completions")

# 4. Disable telemetry to prevent the 401 error
set_tracing_disabled(True)

# Check the API Keys
print(f"Openrouter API Key: {openrouter_api_key[:5]}")
print(f"Tavily API Key: {tavily_api_key[:5]}")

print("OpenAI client configured.")


Openrouter API Key: sk-or
Tavily API Key: tvly-
OpenAI client configured.


In [4]:
def print_markdown(text):
    display(Markdown(text))

In [6]:
import requests
import json  # Import the json module for handling JSON data
from typing_extensions import TypedDict  # Import TypedDict for type hinting
from agents import function_tool

In [7]:
# A TypedDict describes the expected keys and value types for a dict.
# Here: a search "query" (string) and "max_results" (int).

class TavilySearchParams(TypedDict):
    query: str
    max_results: int


# This decorator (from OpenAI Agents SDK) registers the function as a tool
# that an agent can call.

@function_tool
def tavily_search(params: TavilySearchParams) -> str:

    # Tavily search endpoint
    url = "https://api.tavily.com/search"

    # Tell the API we're sending JSON
    headers = {"Content-Type": "application/json"}

    # Build the request body:
    # api_key: your Tavily API key (assumed defined elsewhere as tavily_api_key)
    # query: taken from the params dict
    # max_results: use provided value or default to 3 if missing
    payload = {
        "api_key": tavily_api_key,
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }

    # Send the POST request with JSON body and headers
    response = requests.post(url, json = payload, headers = headers)
    if response.status_code == 200:
        results = response.json().get("results", [])
        summary = "\n".join([f"- {r['title']}: {r['content']}" for r in results])
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"

print("✅ Tavily search tool ready.")

✅ Tavily search tool ready.


# TASK 3. DEFINE TWO AI AGENTS (RESEARCHER & ANALYST)

In [11]:
!pip install pydantic

In [15]:
from pydantic import BaseModel
from agents import Agent, Runner, SQLiteSession

# Let's define our Researcher AI Agent

# Define a Pydantic model for the agent's output.
# This ensures the agent's final output will have a single field: "summary" (string).
# Note that this will be used in both the researcher and analyst agents
class AnalysisSummary(BaseModel):
    summary: str


# Create an AI agent called "Researcher"
researcher_agent = Agent(name = "Researcher",
                         instructions = """
## Context
You are a research agent with access to the Tavily search tool.

## Instruction
Given a user query, use the Tavily search tool to find relevant information and summarize the key findings.

## Input
- A research query from the user.

## Output
- A summary of key findings in a maximum of 5 bullet points.
""",
    model = "nemotron-3-ultra-550b-a55b:free",
    tools = [tavily_search],
    output_type = AnalysisSummary)

print("✅ Researcher AI agent is now ready")

✅ Researcher AI agent is now ready


In [18]:
# Let's run the researcher AI agent
researcher_result = await Runner.run(researcher_agent, "Why is Labubu so popular and what are the rarest Labubu collectibles?")
researcher_result

ValidationError: 1 validation error for InputTokensDetails
cache_write_tokens
  Field required [type=missing, input_value={'cached_tokens': 0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [ ]:
print_markdown(f"### 🤖 Agent’s Answer\n{researcher_result.final_output.summary}")

### 🤖 Agent’s Answer
- Labubu's popularity stems from its cute and quirky design, collectible nature, and the excitement generated by limited availability and blind box releases.
- The toys are popular among Gen Z and Gen Alpha as both accessories and toys, fitting into a long tradition of Asian aesthetics that appeal globally.
- Some Labubu collectibles are sold for high prices on the resale market, with rare editions reaching tens of thousands of dollars.
- Pop Mart, the company behind Labubu, has faced challenges such as product scarcity, resellers, and even product confiscations due to smuggling attempts.
- Limited edition collaborations, like the Vans x Labubu, have further fueled the craze and resale value of rare Labubu collectibles.

In [ ]:
# Let's define our Analyst AI Agent

analyst_agent = Agent(name = "Analyst",
                      instructions = """
## Context
You are an analyst who receives research notes generated by the research agent.

## Instruction
Given the research notes, analyze the content and extract key trends, risks, or insights.

## Input
- Research notes (summaries of findings from the research agent).

## Output
- A concise analysis (no more than 2 paragraphs) highlighting key trends, risks, or insights.
""",
    model = "gpt-4.1-mini",
    output_type = AnalysisSummary,
)

print("✅ Analyst AI agent us now ready")

✅ Analyst AI agent us now ready


**PRACTICE OPPORTUNITY:**  

- **Use the `researcher_agent` and `analyst_agent` to analyze a specific research question or topic.**

  - **Provide a research question (e.g., "What are the latest trends in Batteries for Electric Vehicles?").**
  - **Call the `researcher_agent` with the research question to obtain key findings.**
  - **Pass the researcher's summary to the `analyst_agent` to extract key trends, risks, or insights.**
  - **Show how you would structure the conversation and pass information between the agents.**


# TASK 4: DEFINE A WRITER AI AGENT (EXECUTIVE REPORT GENERATOR)

In [ ]:
# Define a Pydantic model for the agent's output
# This specifies exactly what fields the Writer agent should return

class FinalReport(BaseModel):
    short_summary: str  # A brief 2–3 sentence executive summary
    markdown_report: str    # A detailed report in Markdown format (at least 500 words)
    follow_up_questions: list[str]    # A list of 3–5 suggested follow-up research questions


# Create an AI agent called "Writer"
writer_agent = Agent(
    name = "Writer",
    instructions = """
You are a senior market analyst tasked with generating an executive-level research report.

## Context
You will receive:
- The original user query (the research question or topic of interest).
- Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent' (for key trends, risks, or insights).

You may call these helper tools as needed to gather additional information or clarify findings.

## Instructions
Your job is to synthesize all available information and produce the following outputs:
1. **Executive Summary:** Write a concise summary (2–3 sentences) highlighting the most important findings relevant to the original query.
2. **Detailed Markdown Report:** Compose a comprehensive, well-structured report in markdown format (at least 500 words) that covers key findings, context, implications, and supporting evidence.
3. **Follow-up Research Questions:** Suggest 3–5 thoughtful follow-up questions for further investigation.

## Input
- The original user query.
- Summaries and analyses from the helper tools.

## Output
Return a structured object with:
- `short_summary`: The executive summary.
- `markdown_report`: The detailed markdown report.
- `follow_up_questions`: A list of 3–5 follow-up research questions.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport,
)

print("✅ Writer AI Agent is now ready")

✅ Writer AI Agent is now ready


**PRACTICE OPPORTUNITY:**
- **Rewrite the `writer` agent’s instructions so that it generates all outputs entirely in French (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**
- **Note: We will test this AI Agent in the next task so stay tuned!**

## TASK 5. BUILD A MANAGER FUNCTION TO ORCHESTRATE THE FULL PIPELINE)

In [ ]:
# Create a session to store and share information between agents
session = SQLiteSession("research_agent_practice")

# Define an async function that takes the user's question and runs the agents
async def manager_run(user_query: str):
    # Show the user’s question in a nice markdown format
    print_markdown(f"**User's Request:** {user_query}")

    # Step 1: Run the researcher agent to gather information about the user's query
    researcher_result = await Runner.run(researcher_agent, user_query, session = session)
    research_summary = researcher_result.final_output.summary  # Get the short research summary

    # Step 2: Run the analyst agent to analyze the research findings
    analyst_result = await Runner.run(analyst_agent, research_summary, session = session)
    analysis_summary = analyst_result.final_output.summary  # Get the short analysis summary

    # Step 3: Prepare the combined input for the writer agent
    input_for_writer = (
        f"Original query: {user_query}\n"
        f"Research summary: {research_summary}\n"
        f"Analysis summary: {analysis_summary}"
    )

    # Step 4: Run the writer agent to create the final report
    writer_result = await Runner.run(writer_agent, input_for_writer, session = session)
    final_output: FinalReport = writer_result.final_output  # The final report from the writer

    # Step 5: Display the results in a nice, clear format
    print_markdown("---")
    print_markdown(f"### 📝 **Short Summary:**\n{final_output.short_summary}")  # Quick overview
    print_markdown("\n\n-----------------\n\n")
    print_markdown(f"### 📄 **Full Report (Markdown):**\n{final_output.markdown_report}")  # Detailed report
    print_markdown("\n\n-----------------\n\n")
    print_markdown(
        "### 🔍 **Follow-Up Questions:**\n- "
        + "\n- ".join(final_output.follow_up_questions)  # Any extra questions the user might explore
    )
    print_markdown("\n\n-----------------\n\n")

In [ ]:
await manager_run("What is the public sentiment and expert reviews about the Tesla Cybertruck?")

**User's Request:** What is the public sentiment and expert reviews about the Tesla Cybertruck?

---

### 📝 **Short Summary:**
Le Tesla Cybertruck est largement apprécié par ses propriétaires pour sa technologie avancée, son intérieur spacieux et silencieux, ses performances rapides et sa maniabilité exceptionnelle, en faisant un des meilleurs pick-ups pour un usage quotidien. Bien que certains experts reconnaissent quelques défauts, ils soulignent son innovation et son design unique, tandis que les retours des utilisateurs sur le long terme confirment sa durabilité et un entretien raisonnable. Malgré un certain scepticisme public, l'attrait novateur du Cybertruck maintient une réputation positive solide.



-----------------



### 📄 **Full Report (Markdown):**
# Rapport sur la perception publique et les avis d'experts concernant le Tesla Cybertruck

## Introduction
Le Tesla Cybertruck, lancé avec un style futuriste et une technologie de pointe, suscite depuis son annonce un large éventail de réactions tant du grand public que des spécialistes de l'automobile. Ce rapport synthétise les sentiments exprimés par les propriétaires, les avis des experts, et les tendances observées sur le long terme.

## Sentiment des propriétaires
Les propriétaires de Cybertruck témoignent majoritairement d'une satisfaction élevée. Ils mettent en avant plusieurs atouts clés :

- **Technologie avancée** : Intégration des systèmes de conduite autonome (FSD), commandes digitales avancées.
- **Confort intérieur** : Un habitacle spacieux, silencieux, et ergonomique.
- **Performances** : Rapidité, tenue de route stable avec un comportement dynamique comparable à celui d'une voiture compacte.
- **Facilité d'utilisation** : Maniabilité en ville, facilité de stationnement et autonomie appréciable.

Ces éléments font du Cybertruck un véhicule très apprécié pour un usage quotidien, surpassant souvent les attentes classiques des pick-ups traditionnels.

## Avis des experts
Les experts reconnaissent certains défauts inhérents au design angulaire et au style non conventionnel du Cybertruck, ce qui peut diviser l'opinion publique. Cependant, ils s'accordent généralement sur les points suivants :

- **Innovation** : Le véhicule repousse les limites du concept de pick-up grâce à une structure en acier inoxydable et une motorisation entièrement électrique.
- **Performance technique** : La puissance, l'autonomie et les fonctionnalités intelligentes sont jugées avancées pour le segment.
- **Impact sur le marché** : Le Cybertruck stimule la concurrence et suscite un renouvellement du secteur des véhicules utilitaires.

Ces analyses renforcent l'image d'un véhicule innovant malgré quelques critiques concernant son esthétique ou certaines fonctionnalités.

## Retours à long terme
Les utilisateurs qui possèdent le Cybertruck depuis plus d'un an rapportent une expérience globalement positive, notamment :

- **Durabilité** : Le véhicule conserve ses performances sans usure excessive.
- **Entretien** : Les coûts restent raisonnables et les pannes sont rares.
- **Satisfaction générale** : L'enthousiasme initial ne faiblit pas, avec un maintien d'une forte confiance dans le produit.

Cela confirme la fiabilité du Cybertruck au-delà de la phase de nouveauté.

## Perception publique
Malgré l'émerveillement de nombreux propriétaires, une partie du public demeure sceptique, notamment en raison de :

- Son design atypique qui divise.
- La méfiance envers le constructeur Tesla et sa communication.
- L'incertitude liée à la production et à la disponibilité commerciale.

Cependant, ces réticences n'affaiblissent pas la popularité et la reconnaissance du Cybertruck comme un véhicule emblématique.

## Conclusion
Le Tesla Cybertruck s'impose comme un modèle novateur et apprécié, tant par ses utilisateurs que par des experts qui reconnaissent son caractère avant-gardiste malgré quelques critiques. Sa durabilité et son attractivité devraient lui assurer une place importante sur le marché des pick-ups électriques dans les années à venir.



-----------------



### 🔍 **Follow-Up Questions:**
- Quels sont les principaux défauts techniques identifiés par les experts sur le Tesla Cybertruck ?
- Comment le Cybertruck se compare-t-il aux pick-ups électriques concurrents sur le marché en termes de performances et d'autonomie ?
- Quelle est la perception du Tesla Cybertruck dans les différents marchés internationaux, notamment en Europe et en Asie ?
- Quels sont les impacts environnementaux réels du Cybertruck comparés aux pick-ups traditionnels à essence ou diesel ?
- Quel est le niveau de satisfaction des utilisateurs concernant les services après-vente et le support Tesla pour le Cybertruck ?



-----------------



**PRACTICE OPPORTUNITY:**
- **Run the pipeline again using the `writer` agent with French language report generation capability (from the previous practice opportunity).**

- **Create a new multi-agent team that performs creative advertising:**
  - **Define a `Creative_Director` agent that brainstorms 3–5 ad ideas.**
  - **Define a `Strategist` agent that selects the top 2 ideas and explains the reasoning.**
  - **Define a `Copywriter` agent that writes tweets for each top campaign.**
  - **Set up any necessary tools for these agents.**
  - **Build a pipeline where the output of each agent is passed to the next, similar to the previous example.**
  - **Try running the full pipeline on a sample prompt (e.g., "Launch campaign for a new eco-friendly water bottle in Bali").**

# PRACTICE OPPORTUNITY SOLUTIONS

**PRACTICE OPPORTUNITY SOLUTION:**  

- **Use the `researcher_agent` and `analyst_agent` to analyze a specific research question or topic.**

  - **Provide a research question (e.g., "What are the latest trends in Batteries for Electric Vehicles?").**
  - **Call the `researcher_agent` with the research question to obtain key findings.**
  - **Pass the researcher's summary to the `analyst_agent` to extract key trends, risks, or insights.**
  - **Show how you would structure the conversation and pass information between the agents.**


In [ ]:
# Let's run the researcher AI agent
researcher_result = await Runner.run(researcher_agent, "Electric vehicle batteries trends in 2025")
researcher_result

RunResult(input='Electric vehicle batteries trends in 2025', new_items=[ToolCallItem(agent=Agent(name='Researcher', handoff_description=None, tools=[FunctionTool(name='tavily_search', description='', params_json_schema={'$defs': {'TavilySearchParams': {'properties': {'query': {'title': 'Query', 'type': 'string'}, 'max_results': {'title': 'Max Results', 'type': 'integer'}}, 'required': ['query', 'max_results'], 'title': 'TavilySearchParams', 'type': 'object', 'additionalProperties': False}}, 'properties': {'params': {'$ref': '#/$defs/TavilySearchParams'}}, 'required': ['params'], 'title': 'tavily_search_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000019FD4737560>, strict_json_schema=True, is_enabled=True)], mcp_servers=[], mcp_config={}, instructions='\n## Context\nYou are a research agent with access to the Tavily search tool.\n\n## Instruction\nGiven a user query, use the 

In [ ]:
analyst_result = await Runner.run(analyst_agent, researcher_result.final_output.summary)

In [ ]:
print_markdown(f"### 🤖 Agent’s Answer\n{analyst_result.final_output.summary}")

### 🤖 Agent’s Answer
The EV battery market is expanding rapidly with dominant industry players like LG Energy Solution, Samsung SDI, Panasonic, CATL, and Tesla driving innovation and market dynamics. A notable trend is the rising adoption of lithium iron phosphate (LFP) batteries, especially in the US and EU markets, underpinned by China's significant supply presence. Key technological advances focus on enhancing battery safety through fire-retardant materials and superior cooling mechanisms, projected as vital for 2025. Economically, battery costs are falling steadily, with prices nearing the critical $100/kWh threshold, improving EV affordability relative to conventional vehicles. Sustainability is increasingly emphasized, with lifecycle management strategies such as battery repurposing and recycling becoming integral to environmental impact reduction within the EV battery ecosystem.

**PRACTICE OPPORTUNITY SOLUTION:**
- **Rewrite the `writer` agent’s instructions so that it generates all outputs entirely in French (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**
- **Note: We will test this AI Agent in the next task so stay tuned!**

In [ ]:
writer_agent = Agent(
    name = "Writer",
    instructions = """
You are a senior market analyst tasked with generating an executive-level research report **in French**.

## Context
You will receive:
- The original user query (the research question or topic of interest).
- Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent' (for key trends, risks, or insights).

You may call these helper tools as needed to gather additional information or clarify findings.

## Instructions
Your job is to synthesize all available information and produce the following outputs in **French**:
1. **Executive Summary:** Write a concise summary (2–3 sentences) in French highlighting the most important findings relevant to the original query.
2. **Detailed Markdown Report:** Compose a comprehensive, well-structured report in French in markdown format (at least 500 words) that covers key findings, context, implications, and supporting evidence.
3. **Follow-up Research Questions:** Suggest 3–5 thoughtful follow-up questions in French for further investigation.

## Input
- The original user query.
- Summaries and analyses from the helper tools.

## Output
Return a structured object with:
- `short_summary`: The executive summary in French.
- `markdown_report`: The detailed markdown report in French.
- `follow_up_questions`: A list of 3–5 follow-up research questions in French.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport)


**PRACTICE OPPORTUNITY SOLUTION :**
- **Run the pipeline again using the `writer` agent with French language report generation capability (from the previous practice opportunity).**

- **Create a new multi-agent team that performs creative advertising:**
  - **Define a `Creative_Director` agent that brainstorms 3–5 ad ideas.**
  - **Define a `Strategist` agent that selects the top 2 ideas and explains the reasoning.**
  - **Define a `Copywriter` agent that writes tweets for each top campaign.**
  - **Build a pipeline where the output of each agent is passed to the next, similar to the previous example.**
  - **Try running the full pipeline on a sample prompt (e.g., "Launch campaign for a new eco-friendly water bottle in Bali").**

In [ ]:
session = SQLiteSession("creative_pipeline_practice")

class CreativeIdeas(BaseModel):
    ideas: list[str]

creative_director = Agent(name = "Creative_Director",
                          instructions = """
Context:
You are a creative director at an advertising agency.

Instruction:
Brainstorm 3–5 creative ad campaign ideas for the given product.

Input:
The product name or description.

Output:
A list of 3–5 creative campaign ideas (as bullet points or short phrases).
""",
    model = "gpt-4.1-mini",
    output_type = CreativeIdeas)


In [ ]:

class SelectedCampaigns(BaseModel):
    top_campaigns: list[str]
    reasoning: str

strategist = Agent(name = "Strategist",
                   instructions = """
Context:
You are a marketing strategist evaluating campaign ideas.

Instruction:
From the provided list of campaign ideas, select the top 2 that are most promising and explain your reasoning.

Input:
A list of campaign ideas (one per line).

Output:
A list of the top 2 campaign ideas and a paragraph explaining why they stand out.
""",
    model = "gpt-4.1-mini",
    output_type = SelectedCampaigns,
)


In [ ]:

class TweetCopy(BaseModel):
    tweets: list[str]

copywriter = Agent(name = "Copywriter",
                   instructions = """
Context:
You are a copywriter creating social media content.

Instruction:
For each of the top campaign ideas provided, write 2–3 engaging tweets that could be used to promote the campaign.

Input:
The selected top campaign ideas (one per line).

Output:
A list of 2–3 tweets for each campaign idea.
""",
    model = "gpt-4.1-mini",
    output_type = TweetCopy)


In [ ]:

async def creative_pipeline(product_name: str):
    director_result = await Runner.run(creative_director, product_name, session = session)
    ideas = director_result.final_output.ideas

    strategist_input = "\n".join(ideas)
    strategist_result = await Runner.run(strategist, strategist_input, session = session)
    top_campaigns = strategist_result.final_output.top_campaigns

    tweets_input = "\n".join(top_campaigns)
    copywriter_result = await Runner.run(copywriter, tweets_input, session = session)
    tweets = copywriter_result.final_output.tweets

    print_markdown("### **Top Campaigns:**\n- " + "\n- ".join(top_campaigns))
    print_markdown("### **Tweets:**\n- " + "\n- ".join(tweets))



In [ ]:
await creative_pipeline("Launch campaign for a new eco-friendly water bottle in Bali")

### **Top Campaigns:**
- "Sip Sustainably"
- "From Bali, For Bali"

### **Tweets:**
- Ready to make a difference? Join Bali's top influencers as they embrace our eco-friendly water bottle and say goodbye to single-use plastic! #SipSustainably #EcoBali 🌿💧
- Plastic pollution ends with YOU! Follow local Bali heroes sharing their journeys to reduce waste using our sustainable bottle. Together, let's sip sustainably! #SipSustainably #GreenLiving 🐢🌊
- Clean Bali, one bottle at a time! Join our 'From Bali, For Bali' community clean-up events and get your hands on our eco-friendly water bottle. Let's protect our paradise! #FromBaliForBali #CleanUp🌺
- Give back to the island that gives us so much. Participate in our Bali beach clean-ups and receive our sustainable water bottle as a thank you! Together, we create lasting change. #FromBaliForBali #EcoWarrior 🌴

- **Would love to connect with everyone on LinkedIn: www.linkedin.com/in/dr-ryan-ahmed**

![image.png](attachment:4e31eebc-edb1-4add-be7b-73193e2b8462.png)